In [3]:
!git clone https://github.com/itwasnoteasy/rag-experiments.git

fatal: destination path 'rag-experiments' already exists and is not an empty directory.


In [5]:
%cd rag-experiments

[Errno 2] No such file or directory: 'rag-experiments'
/content/rag-experiments


In [7]:
!pip install -q sentence-transformers chromadb==1.1.0 rank-bm25 transformers torch langfuse google-generativeai pandas tabulate crewai nest_asyncio langgraph Pillow==9.5.0
#

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 86.1 MB/s eta 0:00:00


In [ ]:
!python setup.py

  RAG Experiments — Project Setup Confirmation

Corpus loaded: 20 documents

╭─────────┬──────────────────┬─────────────────────────────────────────────────────────╮
│ ID      │ Category         │ Title                                                   │
├─────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ DI-001  │ device_insurance │ How to File a Device Insurance Claim                    │
│ DI-002  │ device_insurance │ Device Insurance Deductible Amounts by Plan Tier        │
│ DI-003  │ device_insurance │ Coverage Exclusions Under Device Insurance              │
│ DI-004  │ device_insurance │ Device Swap Process After Claim Approval                │
│ DI-005  │ device_insurance │ Water Damage Policy and What's Covered                  │
│ DI-006  │ device_insurance │ Enrolling in Device Insurance After Purchase            │
│ DI-007  │ device_insurance │ Claim Limits and Claim Frequency Policy                 │
│ DI-008  │ device_insurance │ Th

In [ ]:
from rank_bm25 import BM25Okapi
from corpus import CORPUS
from queries import QUERIES

# Index the corpus
tokenized_corpus = [doc["content"].lower().split() for doc in CORPUS]
bm25 = BM25Okapi(tokenized_corpus)

# Run all 10 queries
for q in QUERIES:
    tokens = q["text"].lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[::-1][:3]
    top_docs = [CORPUS[i]["id"] for i in top_idx]
    hit = "✓" if q["expected_doc"] in top_docs else "✗"
    print(f"{hit} [{q['query_type']:12}] {q['id']} | Expected: {q['expected_doc']} | Got: {top_docs}")

✓ [exact_match ] Q01 | Expected: DI-002 | Got: ['DI-002', 'DI-003', 'DI-008']
✓ [exact_match ] Q02 | Expected: DI-004 | Got: ['DI-004', 'SMB-009', 'DI-001']
✓ [semantic    ] Q03 | Expected: DI-005 | Got: ['DI-005', 'DI-006', 'DI-007']
✗ [semantic    ] Q04 | Expected: SMB-009 | Got: ['DI-001', 'SMB-006', 'SMB-007']
✗ [ambiguous   ] Q05 | Expected: DI-006 | Got: ['SMB-008', 'SMB-006', 'SMB-010']
✗ [ambiguous   ] Q06 | Expected: SMB-004 | Got: ['DI-002', 'SMB-007', 'SMB-008']
✗ [context_dep ] Q07 | Expected: DI-002 | Got: ['SMB-004', 'DI-009', 'DI-004']
✗ [context_dep ] Q08 | Expected: SMB-002 | Got: ['DI-003', 'SMB-009', 'DI-002']
✗ [clear_intent] Q09 | Expected: DI-001 | Got: ['DI-007', 'DI-005', 'SMB-007']
✗ [clear_intent] Q10 | Expected: SMB-005 | Got: ['DI-005', 'SMB-001', 'DI-007']


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # fast, good quality

# Embed all documents
doc_texts = [doc["content"] for doc in CORPUS]
doc_embeddings = model.encode(doc_texts, show_progress_bar=True)

# Run all 10 queries
for q in QUERIES:
    q_emb = model.encode(q["text"])
    sims = np.dot(doc_embeddings, q_emb) / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    # print (sims)
    top_idx = sims.argsort()[::-1][:3]
    top_docs = [CORPUS[i]["id"] for i in top_idx]
    hit = "✓" if q["expected_doc"] in top_docs else "✗"
    print(f"{hit} [{q['query_type']:12}] {q['id']} | Expected: {q['expected_doc']} | Got: {top_docs}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ [exact_match ] Q01 | Expected: DI-002 | Got: ['DI-002', 'DI-006', 'DI-003']
✓ [exact_match ] Q02 | Expected: DI-004 | Got: ['DI-004', 'DI-006', 'DI-002']
✓ [semantic    ] Q03 | Expected: DI-005 | Got: ['DI-008', 'DI-005', 'DI-001']
✓ [semantic    ] Q04 | Expected: SMB-009 | Got: ['SMB-002', 'SMB-009', 'SMB-006']
✓ [ambiguous   ] Q05 | Expected: DI-006 | Got: ['DI-006', 'DI-001', 'DI-003']
✗ [ambiguous   ] Q06 | Expected: SMB-004 | Got: ['SMB-003', 'DI-007', 'DI-006']
✓ [context_dep ] Q07 | Expected: DI-002 | Got: ['DI-002', 'DI-003', 'DI-006']
✗ [context_dep ] Q08 | Expected: SMB-002 | Got: ['DI-004', 'SMB-006', 'SMB-001']
✓ [clear_intent] Q09 | Expected: DI-001 | Got: ['DI-001', 'DI-008', 'DI-005']
✓ [clear_intent] Q10 | Expected: SMB-005 | Got: ['SMB-008', 'SMB-007', 'SMB-005']


In [ ]:
# Run both and compare
results = []
for q in QUERIES:
    # BM25
    tokens = q["text"].lower().split()
    bm25_top = [CORPUS[i]["id"] for i in bm25.get_scores(tokens).argsort()[::-1][:6]]

    # Dense
    q_emb = model.encode(q["text"])
    sims = np.dot(doc_embeddings, q_emb) / (np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_emb))
    # Experiment with top1, top2 and top3
    dense_top = [CORPUS[i]["id"] for i in sims.argsort()[::-1][:6]]

    bm25_hit = "✓" if q["expected_doc"] in bm25_top else "✗"
    dense_hit = "✓" if q["expected_doc"] in dense_top else "✗"
    results.append([q["id"], q["query_type"], q["difficulty"], q["expected_doc"],
                    f"{bm25_hit} {bm25_top[0]}", f"{dense_hit} {dense_top[0]}"])

from tabulate import tabulate
print(tabulate(results, headers=["ID","Type","Diff","Expected","BM25 Top1","Dense Top1"], tablefmt="rounded_outline"))

╭──────┬──────────────┬────────┬────────────┬─────────────┬──────────────╮
│ ID   │ Type         │ Diff   │ Expected   │ BM25 Top1   │ Dense Top1   │
├──────┼──────────────┼────────┼────────────┼─────────────┼──────────────┤
│ Q01  │ exact_match  │ easy   │ DI-002     │ ✓ DI-002    │ ✓ DI-002     │
│ Q02  │ exact_match  │ easy   │ DI-004     │ ✓ DI-004    │ ✓ DI-004     │
│ Q03  │ semantic     │ hard   │ DI-005     │ ✓ DI-005    │ ✓ DI-008     │
│ Q04  │ semantic     │ hard   │ SMB-009    │ ✗ DI-001    │ ✓ SMB-002    │
│ Q05  │ ambiguous    │ medium │ DI-006     │ ✓ SMB-008   │ ✓ DI-006     │
│ Q06  │ ambiguous    │ medium │ SMB-004    │ ✗ DI-002    │ ✓ SMB-003    │
│ Q07  │ context_dep  │ hard   │ DI-002     │ ✗ SMB-004   │ ✓ DI-002     │
│ Q08  │ context_dep  │ hard   │ SMB-002    │ ✓ DI-003    │ ✓ DI-004     │
│ Q09  │ clear_intent │ easy   │ DI-001     │ ✗ DI-007    │ ✓ DI-001     │
│ Q10  │ clear_intent │ easy   │ SMB-005    │ ✗ DI-005    │ ✓ SMB-008    │
╰──────┴──────────────┴──

In [ ]:
!python experiment_1_retrieval.py

=== Building dense index (all-MiniLM-L6-v2) ===
Loading weights: 100% 103/103 [00:00<00:00, 6192.85it/s]
Batches: 100% 1/1 [00:02<00:00,  2.22s/it]
  Indexed 20 documents into ChromaDB.

=== Building BM25 index ===
  Indexed 20 documents.

  PER-QUERY COMPARISON: Dense | BM25 | RRF

────────────────────────────────────────────────────────────────────────────────
  Q01 [exact_match] [easy]
  Query   : What is the deductible for a premium smartphone under policy INS-POL-2024?
  Expected: DI-002

╭──────────────────────┬───────────────────────┬──────────────────────╮
│ Dense Top-3          │ BM25 Top-3            │ RRF Top-3            │
├──────────────────────┼───────────────────────┼──────────────────────┤
│ #1 DI-002 (0.7574) ✓ │ #1 DI-002 (14.1318) ✓ │ #1 DI-002 (0.0328) ✓ │
│ #2 DI-006 (0.5591)   │ #2 DI-003 (5.6920)    │ #2 DI-003 (0.0320)   │
│ #3 DI-003 (0.5404)   │ #3 DI-008 (4.6116)    │ #3 DI-006 (0.0161)   │
╰──────────────────────┴───────────────────────┴─────────────────────

In [ ]:
!python experiment_2_reranking.py

=== Loading cross-encoder (ms-marco-MiniLM-L-6-v2) ===
config.json: 100% 794/794 [00:00<00:00, 2.35MB/s]

model.safetensors: downloading bytes:  94% 85.2M/90.9M [00:02<00:00, 31.1MB/s, 7.12MB/s  ]
model.safetensors: downloading bytes: 100% 86.0M/86.0M [00:02<00:00, 29.7MB/s, 7.35MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:02<00:00, 31.3MB/s, 8.48MB/s  ]
Loading weights: 100% 105/105 [00:00<00:00, 5788.90it/s]
tokenizer_config.json: 100% 1.33k/1.33k [00:00<00:00, 3.76MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 14.4MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 56.4MB/s]
special_tokens_map.json: 100% 132/132 [00:00<00:00, 548kB/s]
  Model loaded.

  PER-QUERY DISPLACEMENT: RRF rank → Cross-encoder rank

────────────────────────────────────────────────────────────────────────────────
  Q01 [exact_match] [easy]
  Query   : What is the deductible for a premium smartphone under policy INS-POL-2024?
  Expected: DI-002

╭──────────┬────────────┬───────────┬──────

In [8]:
import os
from google.colab import userdata
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
os.environ["GOOGLE_API_KEY"]      = userdata.get("GEMINI_API_KEY")
os.environ["LANGFUSE_HOST"]      = userdata.get("LANGFUSE_HOST")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") # Added to resolve the error
os.environ["GEMINI_MODEL"] = "gemini-3.1-flash-lite"

In [ ]:
# Run experiments in order — exp 4 loads models from exp 3
!python experiment_4_langfuse.py

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 5.87 KiB | 2.93 MiB/s, done.
From https://github.com/itwasnoteasy/rag-experiments
   0cdef99..5092cd5  claude/eager-ritchie-TC17K -> origin/claude/eager-ritchie-TC17K
Updating 0cdef99..5092cd5
Fast-forward
 results/experiment_9_multiagent_results.json | 238 +++++++++++++++++++++++++++
 1 file changed, 238 insertions(+)
 create mode 100644 results/experiment_9_multiagent_results.json
/content/rag-experiments/experiment_4_langfuse.py:56: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.g

In [ ]:
!python experiment_3_intent.py

=== Loading zero-shot classifier (facebook/bart-large-mnli) ===
config.json: 100% 1.15k/1.15k [00:00<00:00, 4.53MB/s]

model.safetensors: downloading bytes:   3% 50.7M/1.63G [00:02<00:27, 56.9MB/s, 3.90MB/s  ]
model.safetensors: downloading bytes:   4% 69.2M/1.63G [00:03<00:25, 60.2MB/s, 5.62MB/s  ]
model.safetensors: downloading bytes:   5% 80.1M/1.63G [00:03<00:37, 40.8MB/s, 6.51MB/s  ]
model.safetensors: downloading bytes:   6% 94.0M/1.63G [00:03<00:26, 57.6MB/s, 6.96MB/s  ]
model.safetensors: downloading bytes:   7% 116M/1.63G [00:03<00:16, 90.0MB/s, 8.17MB/s  ] 
model.safetensors: downloading bytes:   8% 131M/1.63G [00:03<00:14, 103MB/s, 10.1MB/s  ] 
model.safetensors: downloading bytes:  11% 187M/1.63G [00:04<00:09, 150MB/s, 14.7MB/s  ]
model.safetensors: downloading bytes:  13% 212M/1.63G [00:04<00:10, 137MB/s, 16.7MB/s  ]
model.safetensors: downloading bytes:  65% 1.07G/1.63G [00:08<00:02, 250MB/s, 75.8MB/s  ]
model.safetensors: reconstructing file:  23% 380M/1.63G [00:09<00:31

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
# Optional: os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"
%run experiment_5_comparison.py

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



══════════════════════════════════════════════════════════════════════
  Experiment 5 — Does Retrieval Method Choice Affect Synthesis Quality?
══════════════════════════════════════════════════════════════════════
  Model:    gemini-3.1-flash-lite
  Queries:  Q03, Q07, Q09
  Context:  top-2 docs per retrieval method

══════════════════════════════════════════════════════════════════════
  Building retrieval indexes …
══════════════════════════════════════════════════════════════════════
  [1/3] Dense (bi-encoder) … 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done
  [2/3] BM25 … done
  [3/3] Cross-encoder … 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

done

══════════════════════════════════════════════════════════════════════
  Query Q03 — SEMANTIC / HARD
══════════════════════════════════════════════════════════════════════
  "My handset took a dip in the toilet — will my protection plan cover the repair?"
  Expected: DI-005  |  Notes: 'Handset', 'took a dip', 'toilet' share no tokens with DI-005 which uses 'device…

  Synthesising with Gemini …
    A: Dense … done  (1141.8 ms, 68 tokens out)
    B: BM25 … done  (811.1 ms, 45 tokens out)
    C: RRF+CE … done  (811.6 ms, 46 tokens out)

  Query: "My handset took a dip in the toilet — will my protection plan cover the repair?"

  [A: Dense] (docs: DI-008, DI-005)
       Yes, your protection plan covers accidental liquid immersion,
       including drops in toilets, provided the damage was
       accidental and not due to intentional submersion beyond
       the device’s rated water-resistance level. Please do not
       attempt to charge your device, keep it in a dry
       environm

In [ ]:
!git pull

Updating 768603c..14acaff
Fast-forward
 experiment_8_nli_guardrail.py           | 660 ++++++++++++++++++++++++++++++++
 results/experiment_6_judge_results.json | 358 +++++++++++++++++
 results/experiment_7_hitl_results.json  | 289 ++++++++++++++
 3 files changed, 1307 insertions(+)
 create mode 100644 experiment_8_nli_guardrail.py
 create mode 100644 results/experiment_6_judge_results.json
 create mode 100644 results/experiment_7_hitl_results.json


In [ ]:
%run experiment_6_llm_judge.py


════════════════════════════════════════════════════════════════════════
  Experiment 6 — LLM-as-Judge: Automated RAG Quality Scoring
════════════════════════════════════════════════════════════════════════
  Generator model : gemini-3.1-flash-lite
  Judge model     : gemini-3.1-flash-lite  ← same family (known self-preference bias)
  Dimensions      : Faithfulness, Answer Relevance, Context Precision, Context Recall
  Queries         : all 10

════════════════════════════════════════════════════════════════════════
  Building retrieval indexes …
════════════════════════════════════════════════════════════════════════
  Dense  … 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done
  BM25   … done
  Cross-encoder … 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

done

  Loaded experiment_5 manual scores for 3 queries.

════════════════════════════════════════════════════════════════════════
  [1/10] Q01 — EXACT_MATCH / EASY
════════════════════════════════════════════════════════════════════════
  Query: "What is the deductible for a premium smartphone under policy INS-POL-2024?"
  Context docs: DI-002, DI-003
  Synthesising … done
  Answer: The provided documents do not state the deductible for a premium smartphone under policy INS-POL-2024. Document 1 specifies that policy INS-POL-2024 relates to legacy plan details, but it does not…
  Judging (4 dimensions) …
    Faithfulness           … 1.000  [███████████████]  5965.8 ms
    Answer Relevance       … 1.000  [███████████████]  5712.9 ms
    Context Precision      … 0.500  [████████░░░░░░░]  6292.9 ms
    Context Recall         … 1.000  [███████████████]  5816.0 ms

════════════════════════════════════════════════════════════════════════
  [2/10] Q02 — EXACT_MATCH / EASY
════════════════════

In [ ]:
%run experiment_7_hitl.py


════════════════════════════════════════════════════════════════════════
  Experiment 7 — Human-in-the-Loop Patterns
════════════════════════════════════════════════════════════════════════
  Model  : gemini-3.1-flash-lite
  Part A : confidence-gated escalation for all 10 queries
  Part B : propose-then-confirm for 3 write actions

════════════════════════════════════════════════════════════════════════
  Building retrieval indexes …
════════════════════════════════════════════════════════════════════════
  Dense  … 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done
  BM25   … done
  Cross-encoder … 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

done

════════════════════════════════════════════════════════════════════════
  PART A — Confidence-Gated Escalation (all 10 queries)
════════════════════════════════════════════════════════════════════════
  Thresholds: Faithfulness < 0.7 OR Context Recall < 0.6 → escalate

  [Q01] "What is the deductible for a premium smartphone under policy INS-…"
         faith=1.000  recall=1.000  →  ✓ auto_approved
  [Q02] "How much is the non-return fee if I don't send back my swapped de…"
         faith=1.000  recall=1.000  →  ✓ auto_approved
  [Q03] "My handset took a dip in the toilet — will my protection plan cov…"
         faith=0.900  recall=1.000  →  ✓ auto_approved
  [Q04] "We're moving our company to a new telecom provider and need to ke…"
         faith=1.000  recall=1.000  →  ✓ auto_approved
  [Q05] "How do I add insurance to devices on my account?"
         faith=1.000  recall=1.000  →  ✓ auto_approved
  [Q06] "What happens when I switch to a different plan mid-month?"
         fait

In [ ]:
%run experiment_8_nli_guardrail.py


════════════════════════════════════════════════════════════════════════
  Experiment 8 — NLI Contradiction Guardrail
════════════════════════════════════════════════════════════════════════
  NLI model       : cross-encoder/nli-deberta-v3-base
  Block threshold : contradiction ≥ 0.5
  Adversarial cases: 2

════════════════════════════════════════════════════════════════════════
  Loading NLI model …
════════════════════════════════════════════════════════════════════════
  Loading NLI model: cross-encoder/nli-deberta-v3-base … 

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

done

════════════════════════════════════════════════════════════════════════
  PART 1 — NLI Guardrail on 10 Real Queries
════════════════════════════════════════════════════════════════════════
  PREMISE    = top cross-encoder-ranked chunk from experiment 2
  HYPOTHESIS = synthesised answer from experiment 6
  BLOCK if contradiction probability ≥ 0.5

  [Q01] premise=DI-002    neutral         0.998 [████████████]
  [Q02] premise=DI-004    neutral         0.596 [███████░░░░░]
  [Q03] premise=DI-005    entailment      0.991 [████████████]
  [Q04] premise=SMB-002   entailment      0.998 [████████████]
  [Q05] premise=SMB-008   neutral         0.999 [████████████]
  [Q06] premise=DI-007    neutral         0.995 [████████████]
  [Q07] premise=DI-002    entailment      0.878 [███████████░]
  [Q08] premise=DI-004    neutral         0.999 [████████████]
  [Q09] premise=DI-001    neutral         0.996 [████████████]
  [Q10] premise=SMB-005   entailment      0.955 [███████████░]

═════════════

In [ ]:
%run rag-experiments/experiment_9_multiagent.py

fatal: not a git repository (or any of the parent directories): .git

════════════════════════════════════════════════════════════════════════
  Experiment 9 — Multi-Agent RAG with CrewAI
════════════════════════════════════════════════════════════════════════
  CrewAI model         : gemini/gemini-3.1-flash-lite
  Agents               : Retriever → Analyst → Reviewer
  Revision loop        : max 1 pass
  Hard queries (compare): Q04, Q06, Q08
  Inter-query delay    : 20.0 s  (rate-limit guard)

════════════════════════════════════════════════════════════════════════
  Building retrieval indexes and loading models …
════════════════════════════════════════════════════════════════════════
  [1/4] Dense bi-encoder … 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done
  [2/4] BM25 … done
  [3/4] Cross-encoder re-ranker … 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

done
  [4/4] NLI model … 

...


  ────────────────────────────────────────────────────────────────────
  [10/10] Q10 — clear_intent / easy
  ────────────────────────────────────────────────────────────────────
  Query: "How do I give one of my employees access to manage the company's phone account?"


  Revised: False  |  Latency: 6,646 ms
  Final  : To grant an employee access to manage your account, log in to the admin portal at business.carrier.c…

  ── Retriever output (truncated):
    1. **Document ID: SMB-005**
    **Title:** Admin Portal: Features and Access Management
    **Content:** The SMB admin portal at business.carrier.com/portal
    is the central hub for managing your organization's account. Key
    features include: user and line management (add, suspend, or
    remove lines), device inv…

  ── Analyst output:
    DRAFT ANSWER: To grant an employee access to manage your account,
    log in to the admin portal at business.carrier.com/portal and use
    the role-based access control (RBAC) feature to create a sub-
    administrator account. You can manage these accounts and reset
    credentials under the 'User Management > Admin Accounts' section.
    INFORMATION GAPS: The provided documents do not contain specific
    step-by-step instructions on the exact menu path

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────┬────────────┬──────────────┬───────────┬───────────┬──────────────────────────────────────────────────────────╮
│ Query   │ Type       │ Difficulty   │ Revised   │ Latency   │ Final Answer (truncated)                                 │
├─────────┼────────────┼──────────────┼───────────┼───────────┼──────────────────────────────────────────────────────────┤
│ Q01     │ exact_matc │ easy         │ no        │ 8,344 ms  │ The deductible for a premium smartphone depends on your… │
│ Q02     │ exact_matc │ easy         │ no        │ 5,852 ms  │ If you fail to return your damaged device within 10 bus… │
│ Q03     │ semantic   │ hard         │ no        │ 7,168 ms  │ Yes, your protection plan covers accidental liquid imme… │
│ Q04     │ semantic   │ hard         │ no        │ 7,437 ms  │ You can port your existing business numbers to our netw… │
│ Q05     │ ambiguous  │ medium       │ no        │ 8,258 ms  │ To add insurance to your devices, navigate to the 'Insu… │
│ Q06     │ ambi

7,325 ms
  [2/3] Q06: "What happens when I switch to a different plan mid-month?…"  7,376 ms
  [3/3] Q08: "And how long does that usually take?…"  6,611 ms

════════════════════════════════════════════════════════════════════════
  SIDE-BY-SIDE COMPARISON — 3 Hardest Queries
════════════════════════════════════════════════════════════════════════
  Queries: Q04, Q06, Q08
  These were selected because they are the most likely to expose
  quality differences: semantic paraphrase gap (Q04), ambiguity
  (Q06), and near-empty context-dependent query (Q08).

  ────────────────────────────────────────────────────────────────────
  Q04  [semantic / hard]
  Query: "We're moving our company to a new telecom provider and need to keep our existing contact numbers."

  SINGLE-AGENT (7,325 ms):
    To port your existing numbers to our network, please submit a Port
    Authorization Code (PAC) or account transfer PIN from your current
    carrier via the admin portal. You may port up to 100 numbers
 

In [9]:
%run experiment_9b_langgraph.py


════════════════════════════════════════════════════════════════════════
  Experiment 9b — Multi-Agent RAG with LangGraph
════════════════════════════════════════════════════════════════════════
  LangGraph version    : 1.2.x
  Gemini model         : gemini-3.1-flash-lite
  Nodes                : retriever → analyst → reviewer
  Revision loop        : conditional edge, max 1 pass
  Checkpointer         : MemorySaver (in-memory)
  HITL interrupt       : 2 lowest-scoring queries from exp 6
  Inter-query delay    : 5.0 s

════════════════════════════════════════════════════════════════════════
  Building retrieval indexes …
════════════════════════════════════════════════════════════════════════
  [1/4] Dense bi-encoder … 

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done
  [2/4] BM25 … done
  [3/4] Cross-encoder re-ranker … 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

done

════════════════════════════════════════════════════════════════════════
  Selecting HITL interrupt targets …
════════════════════════════════════════════════════════════════════════
  HITL targets: all faithfulness=1.0 in exp 6 (model generous).
  Falling back to lowest overall-score queries: ['Q01', 'Q02']

════════════════════════════════════════════════════════════════════════
  Building LangGraph state graph …
════════════════════════════════════════════════════════════════════════
  Graph nodes  : ['__start__', 'retriever_node', 'analyst_node', 'reviewer_node']
  Interrupt queries : {'Q02', 'Q01'}

════════════════════════════════════════════════════════════════════════
  Running all 10 queries through the graph …
════════════════════════════════════════════════════════════════════════

  ────────────────────────────────────────────────────────────────────
  [1/10] Q01 — exact_match / easy
  ────────────────────────────────────────────────────────────────────
  Query: "What